# Idealness Prover Comparison

This notebook compares four implementations of idealness verification:

| Implementation | Algorithm | Validation | Output |
|----------------|-----------|------------|--------|
| **exact_prover_v3** | Branch-and-bound | Numerical pre-filter + symbolic solve() | Algebraic conditions |
| **exact_numerical** | Branch-and-bound | Fixed values (L=1, U=9, P=2) | IDEAL/NOT IDEAL |
| **enumerate_symbolic** | Direct enumeration | Numerical pre-filter + SymPy solve() | Algebraic conditions |
| **enumerate_numeric** | Direct enumeration | Numerical test points only | Heuristic conditions |

All analyze the same formulations:
- **SU**: Standard Unary (4 binary variables, equality coupling)
- **RU**: Refined Unary (4 binary variables, inequality coupling)  
- **SB-L**: Simple Binary with Hamming selector (2 binary variables)
- **SB-M**: Simple Binary with Multilinear selector (2 binary + 1 auxiliary)

In [1]:
import time
import pandas as pd
from IPython.display import display, HTML

## 1. Import All Implementations

In [2]:
# Import branch-and-bound symbolic prover (v3)
from exact_prover_v3 import (
    create_SU_formulation as create_SU_v3,
    create_RU_formulation as create_RU_v3,
    create_SBL_formulation as create_SBL_v3,
    create_SBM_formulation as create_SBM_v3,
    ExactBBProver as BBProverV3
)
print("✓ exact_prover_v3 (B&B + symbolic solve)")

✓ exact_prover_v3 (B&B + symbolic solve)


In [3]:
# Import branch-and-bound numerical prover
from exact_numerical import (
    create_SU_formulation as create_SU_bbnum,
    create_RU_formulation as create_RU_bbnum,
    create_SBL_formulation as create_SBL_bbnum,
    create_SBM_formulation as create_SBM_bbnum,
    ExactBBProver as BBProverNum
)
print("✓ exact_numerical (B&B + fixed L=1, U=9, P=2)")

✓ exact_numerical (B&B + fixed L=1, U=9, P=2)


In [4]:
# Import enumeration-based symbolic prover (TRUE symbolic with SymPy solve())
from enumerate_symbolic import (
    create_SU_formulation_symbolic as create_SU_enum_sym,
    create_RU_formulation_symbolic as create_RU_enum_sym,
    create_SBL_formulation_symbolic as create_SBL_enum_sym,
    create_SBM_formulation_symbolic as create_SBM_enum_sym,
    SymbolicIdealnesProver as EnumSymbolicProver
)
print("✓ enumerate_symbolic (enumeration + SymPy solve() for algebraic conditions)")

✓ enumerate_symbolic (enumeration + SymPy solve() for algebraic conditions)


In [5]:
# Import enumeration-based numeric prover (numerical test points only)
from enumerate_numeric import (
    create_SU_formulation_symbolic as create_SU_enum_num,
    create_RU_formulation_symbolic as create_RU_enum_num,
    create_SBL_formulation_symbolic as create_SBL_enum_num,
    create_SBM_formulation_symbolic as create_SBM_enum_num,
    SymbolicIdealnesProver as EnumNumericProver
)
print("✓ enumerate_numeric (enumeration + numerical test points)")

✓ enumerate_numeric (enumeration + numerical test points)


## 2. Run All Provers

In [6]:
# Storage for results
all_results = {}
all_timings = {}

MODELS = ['SB-L', 'SB-M', 'RU', 'SU']

In [7]:
# Run B&B Symbolic (v3)
print("Running B&B Symbolic (exact_prover_v3)...")
v3_factories = {'SB-L': create_SBL_v3, 'SB-M': create_SBM_v3, 'RU': create_RU_v3, 'SU': create_SU_v3}
all_results['bb_sym'] = {}
all_timings['bb_sym'] = {}

for name in MODELS:
    start = time.time()
    prover = BBProverV3(v3_factories[name]())
    _, result = prover.prove()
    all_timings['bb_sym'][name] = time.time() - start
    all_results['bb_sym'][name] = {
        'integral': len(result.always_integral_vertices),
        'conditional': len(result.conditional_vertices),
        'fractional': len(result.always_fractional_vertices),
        'is_ideal': result.is_ideal,
        'is_conditional': result.is_conditionally_ideal,
        'conditions': result.ideal_conditions
    }
    print(f"  {name}: done ({all_timings['bb_sym'][name]:.1f}s)")

Running B&B Symbolic (exact_prover_v3)...
  SB-L: done (0.5s)
  SB-M: done (27.6s)
  RU: done (34.7s)
  SU: done (5.1s)


In [8]:
# Run B&B Numerical
print("Running B&B Numerical (exact_numerical)...")
bbnum_factories = {'SB-L': create_SBL_bbnum, 'SB-M': create_SBM_bbnum, 'RU': create_RU_bbnum, 'SU': create_SU_bbnum}
all_results['bb_num'] = {}
all_timings['bb_num'] = {}

for name in MODELS:
    start = time.time()
    prover = BBProverNum(bbnum_factories[name]())
    _, result = prover.prove()
    all_timings['bb_num'][name] = time.time() - start
    all_results['bb_num'][name] = {
        'integral': len(result.integer_vertices),
        'fractional': len(result.fractional_vertices),
        'is_ideal': result.is_ideal
    }
    print(f"  {name}: done ({all_timings['bb_num'][name]:.1f}s)")

Running B&B Numerical (exact_numerical)...
  SB-L: done (0.1s)
  SB-M: done (47.8s)
  RU: done (44.4s)
  SU: done (10.6s)


In [9]:
# Run Enumeration Symbolic (TRUE symbolic - uses SymPy solve())
print("Running Enumeration Symbolic (enumerate_symbolic)...")
print("  (This uses SymPy's solve() to algebraically derive conditions)")
enum_sym_factories = {'SB-L': create_SBL_enum_sym, 'SB-M': create_SBM_enum_sym, 'RU': create_RU_enum_sym, 'SU': create_SU_enum_sym}
all_results['enum_sym'] = {}
all_timings['enum_sym'] = {}

for name in MODELS:
    start = time.time()
    prover = EnumSymbolicProver(enum_sym_factories[name]())
    result = prover.prove()
    all_timings['enum_sym'][name] = time.time() - start
    all_results['enum_sym'][name] = {
        'integral': result['always_integral'],
        'conditional': result['conditional'],
        'fractional': result['always_fractional'],
        'is_ideal': result['is_ideal'],
        'is_conditional': result['is_conditionally_ideal'],
        'conditions': result['ideal_conditions']
    }
    print(f"  {name}: done ({all_timings['enum_sym'][name]:.1f}s)")

Running Enumeration Symbolic (enumerate_symbolic)...
  (This uses SymPy's solve() to algebraically derive conditions)

SYMBOLIC IDEALNESS PROOF: Simple Binary Hamming (SB-L) [symbolic]

Formulation:
  Variables: 6 (2 binary)
  Constraints: 10 (0 equality)
  Tight inequalities needed per vertex: 6
  Combinations to check: C(10, 6) = 210

Enumeration Statistics:
  Combinations checked: 210
  Rank deficient: 179
  Numerically infeasible: 0
  Symbolically analyzed: 31
  Time: 0.34s

Vertex Classification (SYMBOLIC):
  Always Integral: 30
  Conditional: 0
  Always Fractional: 1

✗ RESULT: NOT IDEAL
  1 always-fractional vertices exist

  Example fractional vertex:
    Tight: frozenset({'a_12y', 'c_12x', 'a_21x', 'a_12x', 'a_21y', 'c_12y'})
    δ_ij = 1/2
    δ_ji = 1/2
  SB-L: done (0.3s)

SYMBOLIC IDEALNESS PROOF: Simple Binary Multilinear (SB-M) [symbolic]

Formulation:
  Variables: 7 (2 binary)
  Constraints: 20 (0 equality)
  Tight inequalities needed per vertex: 7
  Combinations to che

In [10]:
# Run Enumeration Numeric (numerical test points only - FAST)
print("Running Enumeration Numeric (enumerate_numeric)...")
print("  (This uses numerical tests at multiple parameter values)")
enum_num_factories = {'SB-L': create_SBL_enum_num, 'SB-M': create_SBM_enum_num, 'RU': create_RU_enum_num, 'SU': create_SU_enum_num}
all_results['enum_num'] = {}
all_timings['enum_num'] = {}

for name in MODELS:
    start = time.time()
    prover = EnumNumericProver(enum_num_factories[name]())
    result = prover.prove()
    all_timings['enum_num'][name] = time.time() - start
    all_results['enum_num'][name] = {
        'integral': result['always_integral'],
        'conditional': result['conditional'],
        'fractional': result['always_fractional'],
    }
    print(f"  {name}: done ({all_timings['enum_num'][name]:.1f}s)")

Running Enumeration Numeric (enumerate_numeric)...
  (This uses numerical tests at multiple parameter values)

NUMERICAL IDEALNESS ANALYSIS: Simple Binary Hamming (SB-L) [symbolic L, U, P]
Variables: 6 (2 binary)
Constraints: 10 (0 equality)
Need 6 tight inequalities per vertex
  Enumerating ALL C(10, 6) = 210 potential tight sets...
  Using numerical pre-filter at (L=1, U=9, P=2)

Enumeration complete:
  Total combinations: 210
  Numerical pre-filter rejected: 179 (85.2%)
  Symbolically analyzed: 31
  Symbolic rank failures: 0
  Symbolic feasibility failures: 0
  Valid vertices found: 31

Vertex classification:
  Always integral: 30
  Always fractional: 1
  Conditionally integral: 0

Time: 0.51s
  SB-L: done (0.5s)

NUMERICAL IDEALNESS ANALYSIS: Simple Binary Multilinear (SB-M) [symbolic L, U, P]
Variables: 7 (2 binary)
Constraints: 20 (0 equality)
Need 7 tight inequalities per vertex
  Enumerating ALL C(20, 7) = 77520 potential tight sets...
  Using numerical pre-filter at (L=1, U=9,

## 3. Results Comparison

In [11]:
# Build comparison dataframe
comparison_data = []

for model in MODELS:
    bb_sym = all_results['bb_sym'][model]
    bb_num = all_results['bb_num'][model]
    enum_sym = all_results['enum_sym'][model]
    enum_num = all_results['enum_num'][model]
    
    comparison_data.append({
        'Model': model,
        'BB-Sym Int': bb_sym['integral'],
        'BB-Sym Cond': bb_sym['conditional'],
        'BB-Sym Frac': bb_sym['fractional'],
        'BB-Num Int': bb_num['integral'],
        'BB-Num Frac': bb_num['fractional'],
        'Enum-Sym Int': enum_sym['integral'],
        'Enum-Sym Cond': enum_sym['conditional'],
        'Enum-Sym Frac': enum_sym['fractional'],
        'Enum-Num Int': enum_num['integral'],
        'Enum-Num Cond': enum_num['conditional'],
        'Enum-Num Frac': enum_num['fractional'],
    })

df = pd.DataFrame(comparison_data)
print("Vertex Counts (all 4 provers):")
display(df)

Vertex Counts (all 4 provers):


,Model,BB-Sym Int,BB-Sym Cond,BB-Sym Frac,BB-Num Int,BB-Num Frac,Enum-Sym Int,Enum-Sym Cond,Enum-Sym Frac,Enum-Num Int,Enum-Num Cond,Enum-Num Frac
0,SB-L,30,0,1,30,1,30,0,1,30,0,1
1,SB-M,878,156,0,878,156,878,156,0,878,156,0
2,RU,2268,0,0,2268,0,2268,0,0,2268,0,0
3,SU,280,0,0,280,0,280,0,0,280,0,0


In [12]:
# Check agreement between symbolic provers
print("Agreement Check:")
print("="*80)

for model in MODELS:
    bb_sym = all_results['bb_sym'][model]
    bb_num = all_results['bb_num'][model]
    enum_sym = all_results['enum_sym'][model]
    enum_num = all_results['enum_num'][model]
    
    # BB-Sym vs Enum-Sym should match exactly (both do true symbolic analysis)
    sym_match = (bb_sym['integral'] == enum_sym['integral'] and 
                 bb_sym['conditional'] == enum_sym['conditional'] and
                 bb_sym['fractional'] == enum_sym['fractional'])
    
    # BB-Sym vs Enum-Num should also match
    num_match = (bb_sym['integral'] == enum_num['integral'] and 
                 bb_sym['conditional'] == enum_num['conditional'] and
                 bb_sym['fractional'] == enum_num['fractional'])
    
    # Totals
    bb_sym_total = bb_sym['integral'] + bb_sym['conditional'] + bb_sym['fractional']
    bb_num_total = bb_num['integral'] + bb_num['fractional']
    
    print(f"\n{model}:")
    print(f"  BB-Sym vs Enum-Sym: {'✓ MATCH' if sym_match else '✗ MISMATCH'}")
    print(f"  BB-Sym vs Enum-Num: {'✓ MATCH' if num_match else '✗ MISMATCH'}")
    print(f"  Total vertices: BB-Sym={bb_sym_total}, BB-Num={bb_num_total}")

Agreement Check:

SB-L:
  BB-Sym vs Enum-Sym: ✓ MATCH
  BB-Sym vs Enum-Num: ✓ MATCH
  Total vertices: BB-Sym=31, BB-Num=31

SB-M:
  BB-Sym vs Enum-Sym: ✓ MATCH
  BB-Sym vs Enum-Num: ✓ MATCH
  Total vertices: BB-Sym=1034, BB-Num=1034

RU:
  BB-Sym vs Enum-Sym: ✓ MATCH
  BB-Sym vs Enum-Num: ✓ MATCH
  Total vertices: BB-Sym=2268, BB-Num=2268

SU:
  BB-Sym vs Enum-Sym: ✓ MATCH
  BB-Sym vs Enum-Num: ✓ MATCH
  Total vertices: BB-Sym=280, BB-Num=280


## 4. Timing Comparison

In [13]:
timing_data = []
for model in MODELS:
    timing_data.append({
        'Model': model,
        'B&B Symbolic': f"{all_timings['bb_sym'][model]:.2f}s",
        'B&B Numerical': f"{all_timings['bb_num'][model]:.2f}s",
        'Enum Symbolic': f"{all_timings['enum_sym'][model]:.2f}s",
        'Enum Numeric': f"{all_timings['enum_num'][model]:.2f}s",
    })

timing_df = pd.DataFrame(timing_data)
print("Timing (all 4 provers):")
display(timing_df)

# Totals
print(f"\nTotals:")
print(f"  B&B Symbolic:   {sum(all_timings['bb_sym'].values()):.1f}s")
print(f"  B&B Numerical:  {sum(all_timings['bb_num'].values()):.1f}s")
print(f"  Enum Symbolic:  {sum(all_timings['enum_sym'].values()):.1f}s")
print(f"  Enum Numeric:   {sum(all_timings['enum_num'].values()):.1f}s")

Timing (all 4 provers):


,Model,B&B Symbolic,B&B Numerical,Enum Symbolic,Enum Numeric
0,SB-L,0.53s,0.12s,0.34s,0.51s
1,SB-M,27.58s,47.82s,11.02s,30.74s
2,RU,34.71s,44.40s,15.06s,35.78s
3,SU,5.14s,10.63s,1.51s,4.70s



Totals:
  B&B Symbolic:   68.0s
  B&B Numerical:  103.0s
  Enum Symbolic:  27.9s
  Enum Numeric:   71.7s


## 5. Summary

In [14]:
print("="*70)
print("IDEALNESS CONCLUSIONS")
print("="*70)

for model in MODELS:
    bb_sym = all_results['bb_sym'][model]
    
    if bb_sym['is_ideal']:
        print(f"\n{model}: ✓ ALWAYS IDEAL")
        print(f"       All {bb_sym['integral']} vertices are integral for all valid (L, U, P)")
    elif bb_sym['is_conditional']:
        print(f"\n{model}: ◐ CONDITIONALLY IDEAL")
        print(f"       {bb_sym['conditions'][0] if bb_sym['conditions'] else 'Condition unknown'}")
    else:
        print(f"\n{model}: ✗ NOT IDEAL")
        print(f"       Has {bb_sym['fractional']} always-fractional vertices")

print("\n" + "="*70)

IDEALNESS CONCLUSIONS

SB-L: ✗ NOT IDEAL
       Has 1 always-fractional vertices

SB-M: ◐ CONDITIONALLY IDEAL
       Ideal when P ≥ U - L

RU: ✓ ALWAYS IDEAL
       All 2268 vertices are integral for all valid (L, U, P)

SU: ✓ ALWAYS IDEAL
       All 280 vertices are integral for all valid (L, U, P)



## 6. Detailed Symbolic Analysis

This section shows the actual formulas for binary variables at fractional/conditional vertices.

In [15]:
# Store full result objects for detailed analysis
detailed_results = {}

print("Re-running B&B Symbolic to capture full vertex data...")
for name in MODELS:
    prover = BBProverV3(v3_factories[name]())
    _, result = prover.prove()
    detailed_results[name] = result
    print(f"  {name}: captured")

Re-running B&B Symbolic to capture full vertex data...
  SB-L: captured
  SB-M: captured
  RU: captured
  SU: captured


In [16]:
print("="*70)
print("DETAILED SYMBOLIC ANALYSIS")
print("="*70)

for model in MODELS:
    result = detailed_results[model]
    
    print(f"\n{'='*70}")
    print(f"{model}")
    print(f"{'='*70}")
    
    if result.is_ideal:
        print(f"Status: ✓ ALWAYS IDEAL")
        print(f"All {len(result.always_integral_vertices)} vertices have integral binary variables.")
        
    elif result.is_conditionally_ideal:
        print(f"Status: ◐ CONDITIONALLY IDEAL")
        print(f"Condition: {result.ideal_conditions[0]}")
        print(f"\nConditional vertices: {len(result.conditional_vertices)}")
        
        # Show formula from first conditional vertex
        if result.conditional_vertices:
            v = result.conditional_vertices[0]
            print(f"\nExample conditional vertex:")
            print(f"  Tight constraints: {sorted(v.tight_set)}")
            print(f"  Binary variable formulas:")
            for ba in v.binary_analyses:
                print(f"    {ba.var_name} = {ba.symbolic_value}")
                if ba.condition_for_integrality:
                    for param, val, effect in ba.condition_for_integrality:
                        print(f"      → {ba.var_name} = 0 when {param} = {val}")
            
            # Explain the condition
            print(f"\n  Interpretation:")
            print(f"    When P < U - L: δ = (U - L - P)/(U - L + P) ∈ (0, 1) → fractional")
            print(f"    When P = U - L: δ = 0 → integral")
            print(f"    When P > U - L: δ < 0 → infeasible (cut off by δ ≥ 0)")
        
    else:
        print(f"Status: ✗ NOT IDEAL")
        print(f"Always-fractional vertices: {len(result.always_fractional_vertices)}")
        
        # Show the fractional vertex
        if result.always_fractional_vertices:
            v = result.always_fractional_vertices[0]
            print(f"\nFractional vertex:")
            print(f"  Tight constraints: {sorted(v.tight_set)}")
            print(f"  Binary variable values:")
            for ba in v.binary_analyses:
                print(f"    {ba.var_name} = {ba.symbolic_value}")
            print(f"\n  This vertex has δ_ij = δ_ji = 1/2 for ALL parameter values.")
            print(f"  No condition on (L, U, P) can make these integral.")

DETAILED SYMBOLIC ANALYSIS

SB-L
Status: ✗ NOT IDEAL
Always-fractional vertices: 1

Fractional vertex:
  Tight constraints: ['a_12x', 'a_12y', 'a_21x', 'a_21y', 'c_12x', 'c_12y']
  Binary variable values:
    delta_12 = 1/2
    delta_21 = 1/2

  This vertex has δ_ij = δ_ji = 1/2 for ALL parameter values.
  No condition on (L, U, P) can make these integral.

SB-M
Status: ◐ CONDITIONALLY IDEAL
Condition: Ideal when P ≥ U - L

Conditional vertices: 156

Example conditional vertex:
  Tight constraints: ['a_ijy', 'a_jix', 'a_jiy', 'b_jix', 'c_ijx', 'mc2', 'mc3']
  Binary variable formulas:
    delta_12 = (-L - P + U)/(-L + P + U)
      → delta_12 = 0 when P = -L + U
    delta_21 = (-L - P + U)/(-L + P + U)
      → delta_21 = 0 when P = -L + U

  Interpretation:
    When P < U - L: δ = (U - L - P)/(U - L + P) ∈ (0, 1) → fractional
    When P = U - L: δ = 0 → integral
    When P > U - L: δ < 0 → infeasible (cut off by δ ≥ 0)

RU
Status: ✓ ALWAYS IDEAL
All 2268 vertices have integral binary va

In [17]:
# Verify enumerate_numeric produces the same formulas
print("="*70)
print("CROSS-VALIDATION: Enumerate Numeric Formulas")
print("="*70)

from sympy import symbols
P_sym = symbols('P', real=True, positive=True)

for model in ['SB-L', 'SB-M']:  # Only show non-ideal ones
    prover = EnumNumericProver(enum_num_factories[model]())
    result = prover.prove(verbose=False)
    
    print(f"\n{model}:")
    if prover.always_fractional:
        v = prover.always_fractional[0]
        print(f"  Fractional vertex:")
        for var_name, val in v.binary_values:
            print(f"    {var_name} = {val}")
    elif prover.conditional:
        v = prover.conditional[0]
        print(f"  Conditional vertex:")
        for var_name, val in v.binary_values:
            print(f"    {var_name} = {val}")
        if v.conditions_for_integrality:
            print(f"  Conditions for integrality:")
            for var_name, val_type, cond in v.conditions_for_integrality:
                cond_val = cond.get(P_sym, cond)
                print(f"    {var_name} {val_type} when P = {cond_val}")
    else:
        print(f"  All vertices integral")

CROSS-VALIDATION: Enumerate Numeric Formulas

SB-L:
  Fractional vertex:
    δ_ij = 1/2
    δ_ji = 1/2

SB-M:
  Conditional vertex:
    δ_12 = (-L - P + U)/(-L + P + U)
    δ_21 = (-L - P + U)/(-L + P + U)
  Conditions for integrality:
    δ_12 =0 when P = -L + U
    δ_21 =0 when P = -L + U


## 7. Generate HTML Proof Reports (Optional)

In [18]:
# Optional: Generate HTML proof reports from exact_prover_v3
# Uncomment to run

# from exact_prover_v3 import run_all
# import os
# 
# output_dir = './proof_reports/'
# os.makedirs(output_dir, exist_ok=True)
# 
# html_results = run_all(verbose=False, html_dir=output_dir)
# 
# print("HTML reports generated:")
# for model in ['SU', 'RU', 'SBL', 'SBM']:
#     print(f"  - {output_dir}{model}_proof.html")